# 02 Plan A 3DGS (supports `colmap_108` and `colmap_full`)

目标：

1. 从 `shared/images_512/` 与子集/全量图像准备 Plan A 工作区
2. 根据 `MODE` 选择 `colmap_108` 或 `colmap_full`
3. 跑完整 COLMAP pipeline：feature extraction / matching / mapper / undistort
4. 整理成标准 3DGS 输入结构 `gs_scene`

这本 notebook 支持两种模式：
- `MODE = 'colmap_108'`：使用 `subset_108.txt`
- `MODE = 'colmap_full'`：使用 `shared/images_512/` 全量图像



In [122]:
from pathlib import Path
import shutil
import subprocess

CV_ROOT = Path('/home/bzhang512/CV_Project')
DATASET_ROOT = CV_ROOT / 'dataset'

SCENE = 'Re10k-1'
#SCENE = 'DL3DV-2'
#SCENE = '405841'
MODE = 'colmap_108'   # 可选: 'colmap_108' 或 'colmap_full'
SUBSET_NAME = 'subset_108.txt'
OVERWRITE = False

SCENE_ROOT = DATASET_ROOT / SCENE
SHARED_ROOT = SCENE_ROOT / 'part1' / 'shared'
IMAGES_512_DIR = SHARED_ROOT / 'images_512'
SUBSETS_DIR = SHARED_ROOT / 'subsets'
PLAN_A_ROOT = SCENE_ROOT / 'part1' / 'planA'
WORK_DIR = PLAN_A_ROOT / MODE

assert MODE in {'colmap_108', 'colmap_full'}, f'Unsupported MODE: {MODE}'

print('WORK_DIR =', WORK_DIR)
print('MODE =', MODE)


WORK_DIR = /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108
MODE = colmap_108


## 1. 依赖和环境检查

In [123]:
def run(cmd, cwd=None):
    print('\n[CMD]', ' '.join(cmd))
    return subprocess.run(cmd, cwd=cwd, check=True)

run(['which', 'colmap'])


[CMD] which colmap
/home/bzhang512/miniconda3/envs/3dgs/bin/colmap


CompletedProcess(args=['which', 'colmap'], returncode=0)

## 2. 准备 COLMAP 工作区（使用 subset_108）

In [124]:
for d in [WORK_DIR, WORK_DIR / 'images', WORK_DIR / 'sparse', WORK_DIR / 'undistorted', WORK_DIR / 'gs_scene']:
    d.mkdir(parents=True, exist_ok=True)

if MODE == 'colmap_108':
    subset_file = SUBSETS_DIR / SUBSET_NAME
    selected_names = [line.strip() for line in subset_file.read_text().splitlines() if line.strip()]
    selected_paths = [IMAGES_512_DIR / name for name in selected_names]
else:
    selected_paths = sorted([p for p in IMAGES_512_DIR.iterdir() if p.is_file()])

for src in selected_paths:
    dst = WORK_DIR / 'images' / src.name
    if dst.exists() and not OVERWRITE:
        continue
    shutil.copy2(src, dst)

print('prepared images:', len(selected_paths))
print('first 5:', [p.name for p in selected_paths[:5]])


prepared images: 108
first 5: ['00000.png', '00003.png', '00005.png', '00008.png', '00010.png']


## 3. Feature extraction

In [125]:
run([
    'colmap', 'feature_extractor',
    '--database_path', str(WORK_DIR / 'database.db'),
    '--image_path', str(WORK_DIR / 'images'),
    '--ImageReader.camera_model', 'SIMPLE_PINHOLE',
    '--ImageReader.single_camera', '1',
    '--SiftExtraction.max_num_features', '4096',
])


[CMD] colmap feature_extractor --database_path /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/database.db --image_path /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/images --ImageReader.camera_model SIMPLE_PINHOLE --ImageReader.single_camera 1 --SiftExtraction.max_num_features 4096


I20260327 23:26:52.807485 140661602422784 misc.cc:44] 
Feature extraction
I20260327 23:26:52.814691 140661126758400 sift.cc:720] Creating SIFT GPU feature extractor
I20260327 23:26:52.815008 140661118365696 sift.cc:720] Creating SIFT GPU feature extractor
I20260327 23:26:53.155786 140661109972992 feature_extraction.cc:260] Processed file [1/108]
I20260327 23:26:53.155850 140661109972992 feature_extraction.cc:263]   Name:            00003.png
I20260327 23:26:53.155857 140661109972992 feature_extraction.cc:272]   Dimensions:      512 x 512
I20260327 23:26:53.155861 140661109972992 feature_extraction.cc:275]   Camera:          #1 - SIMPLE_PINHOLE
I20260327 23:26:53.155868 140661109972992 feature_extraction.cc:278]   Focal Length:    614.40px
I20260327 23:26:53.155882 140661109972992 feature_extraction.cc:282]   Features:        1342 (SIFT)
I20260327 23:26:53.299623 140661109972992 feature_extraction.cc:260] Processed file [2/108]
I20260327 23:26:53.299683 140661109972992 feature_extractio

CompletedProcess(args=['colmap', 'feature_extractor', '--database_path', '/home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/database.db', '--image_path', '/home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/images', '--ImageReader.camera_model', 'SIMPLE_PINHOLE', '--ImageReader.single_camera', '1', '--SiftExtraction.max_num_features', '4096'], returncode=0)

## 4. Sequential matching

In [126]:
run([
    'colmap', 'sequential_matcher',
    '--database_path', str(WORK_DIR / 'database.db'),
    '--SequentialMatching.overlap', '10',
    '--SequentialMatching.quadratic_overlap', '1',
])


[CMD] colmap sequential_matcher --database_path /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/database.db --SequentialMatching.overlap 10 --SequentialMatching.quadratic_overlap 1


I20260327 23:26:54.902584 139948632440832 misc.cc:44] 
Feature matching & geometric verification
I20260327 23:26:55.155514 139948409470976 feature_matching_utils.cc:78] Bind FeatureMatcherWorker to GPU device 1
I20260327 23:26:55.156139 139948409470976 sift.cc:1446] Creating SIFT GPU feature matcher
I20260327 23:26:55.162428 139948417863680 feature_matching_utils.cc:78] Bind FeatureMatcherWorker to GPU device 0
I20260327 23:26:55.162466 139948417863680 sift.cc:1446] Creating SIFT GPU feature matcher
I20260327 23:26:55.163212 139948632440832 pairing.cc:412] Generating sequential image pairs...
I20260327 23:26:55.163942 139948632440832 pairing.cc:470] Processing image [1/108]
I20260327 23:26:55.178104 139948632440832 feature_matching.cc:117] in 0.014s
I20260327 23:26:55.178141 139948632440832 pairing.cc:470] Processing image [2/108]
I20260327 23:26:55.187863 139948632440832 feature_matching.cc:117] in 0.010s
I20260327 23:26:55.187893 139948632440832 pairing.cc:470] Processing image [3/10

CompletedProcess(args=['colmap', 'sequential_matcher', '--database_path', '/home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/database.db', '--SequentialMatching.overlap', '10', '--SequentialMatching.quadratic_overlap', '1'], returncode=0)

## 5. Mapper

In [127]:
run([
    'colmap', 'mapper',
    '--database_path', str(WORK_DIR / 'database.db'),
    '--image_path', str(WORK_DIR / 'images'),
    '--output_path', str(WORK_DIR / 'sparse'),
])


[CMD] colmap mapper --database_path /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/database.db --image_path /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/images --output_path /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/sparse


I20260327 23:26:56.228400 140261365452800 incremental_pipeline.cc:264] Loading database
I20260327 23:26:56.233759 140261365452800 database_cache.cc:67] Loading rigs...
I20260327 23:26:56.233956 140261365452800 database_cache.cc:77]  1 in 0.000s
I20260327 23:26:56.234010 140261365452800 database_cache.cc:85] Loading cameras...
I20260327 23:26:56.234076 140261365452800 database_cache.cc:103]  1 in 0.000s
I20260327 23:26:56.234120 140261365452800 database_cache.cc:111] Loading frames...
I20260327 23:26:56.234323 140261365452800 database_cache.cc:126]  108 in 0.000s
I20260327 23:26:56.234373 140261365452800 database_cache.cc:134] Loading matches...
I20260327 23:26:56.243609 140261365452800 database_cache.cc:139]  629 in 0.009s
I20260327 23:26:56.243749 140261365452800 database_cache.cc:155] Loading images...
I20260327 23:26:56.257045 140261365452800 database_cache.cc:239]  108 in 0.013s (connected 108)
I20260327 23:26:56.257275 140261365452800 database_cache.cc:250] Building correspondence

CompletedProcess(args=['colmap', 'mapper', '--database_path', '/home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/database.db', '--image_path', '/home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/images', '--output_path', '/home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/sparse'], returncode=0)

## 6. Model inspection

In [128]:
# 列出 sparse/ 下所有重建结果目录（0,1,2,...），供下一 cell 手动选择
sparse_root = WORK_DIR / 'sparse'
recon_dirs = sorted([p for p in sparse_root.iterdir() if p.is_dir()]) if sparse_root.exists() else []

print('sparse_root =', sparse_root)
print('found recon dirs =', [p.name for p in recon_dirs])

for p in recon_dirs:
    print('--- analyzer for', p.name, '---')
    try:
        run(['colmap', 'model_analyzer', '--path', str(p)])
    except Exception as e:
        print('analyzer failed for', p, e)


sparse_root = /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/sparse
found recon dirs = ['0']
--- analyzer for 0 ---

[CMD] colmap model_analyzer --path /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/sparse/0


I20260327 23:28:30.333959 139835772239872 model.cc:451] Rigs: 1
I20260327 23:28:30.334354 139835772239872 model.cc:452] Cameras: 1
I20260327 23:28:30.334367 139835772239872 model.cc:453] Frames: 108
I20260327 23:28:30.334373 139835772239872 model.cc:454] Registered frames: 108
I20260327 23:28:30.334379 139835772239872 model.cc:456] Images: 108
I20260327 23:28:30.334383 139835772239872 model.cc:457] Registered images: 108
I20260327 23:28:30.334395 139835772239872 model.cc:459] Points: 6119
I20260327 23:28:30.334400 139835772239872 model.cc:460] Observations: 107475
I20260327 23:28:30.334410 139835772239872 model.cc:462] Mean track length: 17.564144
I20260327 23:28:30.334437 139835772239872 model.cc:464] Mean observations per image: 995.138889
I20260327 23:28:30.334451 139835772239872 model.cc:467] Mean reprojection error: 0.645599px


## 7. Undistort and organize gs_scene

In [ ]:
# 选择 sparse model 后，先清理旧 gs_scene，再做 undistort，并规范成 sparse/0
import shutil

sparse_root = WORK_DIR / 'sparse'
recon_dirs = sorted([p for p in sparse_root.iterdir() if p.is_dir()]) if sparse_root.exists() else []
assert recon_dirs, f'No sparse recon dirs found under {sparse_root}'

# 你可以手动改这里，比如 recon_dirs[1]
SPARSE_MODEL_DIR = recon_dirs[0]
print('Using sparse model dir =', SPARSE_MODEL_DIR)

gs_scene_dir = WORK_DIR / 'gs_scene'
if gs_scene_dir.exists():
    shutil.rmtree(gs_scene_dir)
(gs_scene_dir / 'sparse' / '0').mkdir(parents=True, exist_ok=True)

run([
    'colmap', 'image_undistorter',
    '--image_path', str(WORK_DIR / 'images'),
    '--input_path', str(SPARSE_MODEL_DIR),
    '--output_path', str(gs_scene_dir),
    '--output_type', 'COLMAP',
])

# 规范化到 3DGS / Scaffold 期望的 sparse/0 结构
sparse_dir = gs_scene_dir / 'sparse'
files = [p for p in sparse_dir.iterdir() if p.is_file()] if sparse_dir.exists() else []
if files:
    sparse0 = sparse_dir / '0'
    sparse0.mkdir(parents=True, exist_ok=True)
    for p in files:
        dst = sparse0 / p.name
        if dst.exists():
            dst.unlink()
        shutil.move(str(p), str(dst))

print('Plan A gs_scene ready at', gs_scene_dir)
print('Expected 3DGS sparse/0 exists =', (gs_scene_dir / 'sparse' / '0').exists())
print('sparse/0 contents =', sorted([p.name for p in (gs_scene_dir / 'sparse' / '0').iterdir()]) if (gs_scene_dir / 'sparse' / '0').exists() else [])
print('MODE completed =', MODE)


Using sparse model dir = /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/sparse/0

[CMD] colmap image_undistorter --image_path /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/images --input_path /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/sparse/0 --output_path /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/gs_scene --output_type COLMAP


I20260327 23:29:45.061698 140547210825728 misc.cc:44] 
Reading reconstruction
I20260327 23:29:45.159269 140547210825728 image.cc:374] => Reconstruction with 108 images and 6119 points
I20260327 23:29:45.159500 140547210825728 misc.cc:44] 
Image undistortion
I20260327 23:29:45.169354 140543796264960 undistortion.cc:239] Undistorted image found; copying to location: /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/gs_scene/images/00003.png
I20260327 23:29:45.169464 140543695552512 undistortion.cc:239] Undistorted image found; copying to location: /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/gs_scene/images/00005.png
I20260327 23:29:45.169558 140543720730624 undistortion.cc:239] Undistorted image found; copying to location: /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/gs_scene/images/00008.png
I20260327 23:29:45.169665 140543762694144 undistortion.cc:239] Undistorted image found; copying to location: /home/bzhang512/CV_Project/dat

Plan A gs_scene ready at /home/bzhang512/CV_Project/dataset/Re10k-1/part1/planA/colmap_108/gs_scene
Expected 3DGS sparse/0 exists = True
sparse/0 contents = ['cameras.bin', 'frames.bin', 'images.bin', 'points3D.bin', 'rigs.bin']
MODE completed = colmap_108


: 